# Лабораторная работа №2 — Линейная, гребневая и лассо-регрессия
**Курс:** Искусственный интеллект  
**Датасет:** *kc_house_data.csv*  
**Цель:** предсказать цену дома и сравнить модели: `LinearRegression`, `Ridge`, `Lasso`.


## 1. Загрузка библиотек и данных


In [ ]:

import numpy as np, pandas as pd, matplotlib.pyplot as plt, seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression, Ridge, Lasso
from sklearn.metrics import r2_score, mean_squared_error
from sklearn.pipeline import Pipeline

plt.rcParams['figure.figsize']=(10,6); plt.rcParams['axes.grid']=True
sns.set(style="whitegrid", context="notebook")
RANDOM_STATE=42


In [ ]:

# Загрузка CSV
csv_path = "kc_house_data.csv"  # замените при необходимости
df = pd.read_csv(csv_path)
print("Размерность:", df.shape)
display(df.head()); display(df.info()); display(df.describe(include='all').T)


## 2. Предобработка данных


In [ ]:

# Пропуски/дубликаты
display(df.isna().sum().sort_values(ascending=False)[lambda s: s>0])
print("Дубликаты:", df.duplicated().sum())

# Дата -> datetime и удаление неинформативных
df['date']=pd.to_datetime(df['date'])
drop_cols = ['id','date','zipcode','lat','long','sqft_basement']
df = df.drop(columns=[c for c in drop_cols if c in df.columns])
print("После удаления:", df.shape)


## 3. Анализ и удаление выбросов


In [ ]:

def trim_by_percentile(data, col, low_q=0.01, high_q=0.99):
    lo, hi = data[col].quantile([low_q, high_q])
    return data[(data[col]>=lo)&(data[col]<=hi)]

n0=len(df)
df=trim_by_percentile(df,'price',0.01,0.99)
if 'sqft_living' in df.columns:
    df=trim_by_percentile(df,'sqft_living',0.01,0.99)
print("Удалено:", n0-len(df))

fig,ax=plt.subplots(1,2,figsize=(14,5))
sns.histplot(df['price'],bins=50,kde=True,ax=ax[0]); ax[0].set_title("Распределение цены")
sns.histplot(df['sqft_living'],bins=50,kde=True,ax=ax[1]); ax[1].set_title("Распределение жилой площади")
plt.tight_layout(); plt.show()


## 4. Стандартизация признаков


In [ ]:

target_col='price'
num_cols=[c for c in df.columns if c!=target_col]
display(df[num_cols].describe().T.head(10))


## 5. Выделение новых признаков


In [ ]:

df_feat=df.copy()
df_feat['price_per_sqft']=df_feat['price']/np.where(df_feat['sqft_living']==0,np.nan,df_feat['sqft_living'])
df_feat['house_age']=2015-df_feat['yr_built']
df_feat['sqft_ratio']=df_feat['sqft_living']/np.where(df_feat['sqft_lot']==0,np.nan,df_feat['sqft_lot'])
df_feat['price_per_sqft']=df_feat['price_per_sqft'].fillna(df_feat['price_per_sqft'].median())
df_feat['sqft_ratio']=df_feat['sqft_ratio'].fillna(df_feat['sqft_ratio'].median())

corr_with_price=df_feat.corr(numeric_only=True)['price'].sort_values(ascending=False)
display(corr_with_price.to_frame('corr_with_price'))


## 6. Train/Test split


In [ ]:

X=df_feat[[c for c in df_feat.columns if c!='price']].copy()
y=df_feat['price'].copy()
X_train,X_test,y_train,y_test=train_test_split(X,y,test_size=0.2,random_state=42)
X_train.shape,X_test.shape


## 7. Модели: Linear / Ridge / Lasso


In [ ]:

def rmse(y_true,y_pred): return float(np.sqrt(mean_squared_error(y_true,y_pred)))
def fit_eval(name, model):
    pipe=Pipeline([('scaler',StandardScaler()),('reg',model)])
    pipe.fit(X_train,y_train)
    ytr, yte = pipe.predict(X_train), pipe.predict(X_test)
    return {'model':name,'r2_train':r2_score(y_train,ytr),'r2_test':r2_score(y_test,yte),
            'rmse_train':rmse(y_train,ytr),'rmse_test':rmse(y_test,yte),'pipe':pipe}

res=[]
res.append(fit_eval("LinearRegression", LinearRegression()))
res.append(fit_eval("Ridge (alpha=3)", Ridge(alpha=3, random_state=RANDOM_STATE)))
res.append(fit_eval("Lasso (alpha=120)", Lasso(alpha=120, random_state=RANDOM_STATE, max_iter=10000)))

res_df=pd.DataFrame([{k:v for k,v in r.items() if k!='pipe'} for r in res]).set_index('model')
display(res_df.style.format({'r2_train':'{:.3f}','r2_test':'{:.3f}','rmse_train':'{:,.0f}','rmse_test':'{:,.0f}'}))


## 8. Коэффициенты моделей


In [ ]:

coef_table=pd.DataFrame(index=X.columns)
for r in res:
    name=r['model']; reg=r['pipe'].named_steps['reg']
    coef_table[name]=reg.coef_
coef_table=coef_table.reindex(coef_table.index[np.argsort(-np.abs(coef_table['LinearRegression'].values))])
display(coef_table.head(20).style.format('{:.3f}'))


## 9. Визуализация предсказаний и остатков


In [ ]:

best=max(res,key=lambda d:d['r2_test'])
best_name,best_pipe=best['model'],best['pipe']
print("Лучшая модель:", best_name)
y_pred=best_pipe.predict(X_test); residuals=y_test-y_pred

plt.figure(figsize=(7,6))
plt.scatter(y_test,y_pred,alpha=0.5); m=min(min(y_test),min(y_pred)); M=max(max(y_test),max(y_pred))
plt.plot([m,M],[m,M],'--')
plt.xlabel('Фактическая цена'); plt.ylabel('Предсказанная цена'); plt.title(f'Факт vs Прогноз — {best_name}')
plt.tight_layout(); plt.show()

fig,ax=plt.subplots(1,2,figsize=(14,5))
sns.histplot(residuals,bins=50,kde=True,ax=ax[0]); ax[0].set_title('Гистограмма остатков (test)'); ax[0].set_xlabel('residual')
ax[1].scatter(y_pred,residuals,alpha=0.5); ax[1].axhline(0,ls='--'); ax[1].set_title('Остатки vs Предсказания'); ax[1].set_xlabel('y_pred'); ax[1].set_ylabel('residual')
plt.tight_layout(); plt.show()


## 10. Выводы
- **Ridge** (L2) уменьшает разброс коэффициентов и часто даёт более устойчивое качество при мультиколлинеарности.  
- **Lasso** (L1) может занулять часть коэффициентов (отбор признаков), повышая интерпретируемость, но при высокой `alpha` — снижая R²/повышая RMSE.  
- Лучшей признаётся модель с **макс. R² (test)** и **мин. RMSE (test)** (см. таблицу).  
- Наиболее значимые признаки см. в таблице коэффициентов (топ-20).
